Описание задачи:

В нашей сети около 100 аптек. Нам нужно единоразово загрузить файл в 1С, но для этого надо сделать сравнение реализации по аптекам и выгрузки из СБИСа.

1. Выгрузка из СБИС находится в папке Входящие.
2. Загрузите все csv файлы в один датафрейм с определенным набором столбцов.

Сравнение реализации по аптекам: Файлы с выгрузкой из аптек находятся в папке Аптеки.

Загружайте на обработку по одному файлу. Содержимое каждого файла загружайте в датафрейм Pandas.

Столбцы датафрейма:

"Номер счет-фактуры"
"Сумма счет-фактуры"
"Дата счет-фактуры"
"Сравнение дат"
Если "Поставщик" - ЕАПТЕКА, то к "Номер накладной" нужно добавить /15.

Нужно найти все записи в выгрузке из СБИСа по данному номеру накладной: из найденных строк нужно оставить только те, которые имеют один из типов документа: ["СчФктр", "УпдДоп", "УпдСчфДоп", "ЭДОНакл"]. Если найдено - нужно сохранить значения Номер, Сумма и Дата

Дату нужно представить в формате 25.05.2021

В столбец Сравнение дат помещаем "Не совпадает!", если найденная дата и дата накладной отличаются. Иначе - пустая строка.

In [98]:
import pandas as pd
import glob

In [99]:
sbis_1 = pd.read_csv('C:/Work/Analytics/Тест/Выгрузка данных в один источник/Входящие/Входящие 01.csv', encoding='windows-1251', sep=';')

sbis_1.head(5)

,Дата,Номер,Сумма,Статус,Примечание,Комментарий,Контрагент,ИНН/КПП,Организация,ИНН/КПП.1,...,Время,Тип пакета,Идентификатор пакета,Запущено в обработку,Получено контрагентом,Завершено,Увеличение суммы,НДС,Уменьшение суммы,НДС.1
0,30.09.21,БРН00709545,"4 056,46",Выполнение завершено успешно,NaN,"БРН00709545 на сумму 4 056,46р.","Пульс Брянск, ООО",3255510243 / 325701001,ООО Рога и Копыта,4025419873 / 402501001,...,23:40:56,ДокОтгрВх,76210df0-7c48-4e6d-ad2b-6cbe2f3b7722,30.09.21 23:40,01.10.21 08:39,01.10.21 14:59,"0,00","0,00","0,00","0,00"
1,30.09.21,БРН00709545,"4 056,46",Выполнение завершено успешно,NaN,"БРН00709545 на сумму 4 056,46р.","Пульс Брянск, ООО",3255510243 / 325701001,ООО Рога и Копыта,4025419873 / 402501001,...,23:40:56,ДокОтгрВх,76210df0-7c48-4e6d-ad2b-6cbe2f3b7722,NaN,NaN,NaN,"0,00","0,00","0,00","0,00"
2,30.09.21,БРН00709545,"0,00",Выполнение завершено успешно,NaN,"БРН00709545 на сумму 4 056,46р.","Пульс Брянск, ООО",3255510243 / 325701001,ООО Рога и Копыта,4025419873 / 402501001,...,23:40:56,ДокОтгрВх,76210df0-7c48-4e6d-ad2b-6cbe2f3b7722,NaN,NaN,NaN,"0,00","0,00","0,00","0,00"
3,30.09.21,БРН00709520,"54 705,65",Выполнение завершено успешно,NaN,"БРН00709520 на сумму 54 705,65р.","Пульс Брянск, ООО",3255510243 / 325701001,ООО Рога и Копыта,4025419873 / 402501001,...,23:34:51,ДокОтгрВх,1cfdcbd4-a43b-4543-90b6-59d0e6095b94,30.09.21 23:34,01.10.21 08:39,01.10.21 14:59,"0,00","0,00","0,00","0,00"
4,30.09.21,БРН00709520,"54 705,65",Выполнение завершено успешно,NaN,"БРН00709520 на сумму 54 705,65р.","Пульс Брянск, ООО",3255510243 / 325701001,ООО Рога и Копыта,4025419873 / 402501001,...,23:34:51,ДокОтгрВх,1cfdcbd4-a43b-4543-90b6-59d0e6095b94,NaN,NaN,NaN,"0,00","0,00","0,00","0,00"


In [100]:
sbis_1.columns

Index(['Дата', 'Номер', 'Сумма', 'Статус', 'Примечание', 'Комментарий',
       'Контрагент', 'ИНН/КПП', 'Организация', 'ИНН/КПП.1', 'Тип документа',
       'Имя файла', 'Дата.1', 'Номер.1', 'Сумма.1', 'Сумма НДС',
       'Ответственный', 'Подразделение', 'Код', 'Дата.2', 'Время',
       'Тип пакета', 'Идентификатор пакета', 'Запущено в обработку',
       'Получено контрагентом', 'Завершено', 'Увеличение суммы', 'НДС',
       'Уменьшение суммы', 'НДС.1'],
      dtype='str')

In [101]:
sbis_1['Тип документа'].value_counts()

Тип документа
ЭДОРеестрСертификатов    29333
СчФктр                   20740
ПротЦен                  17730
УпдДоп                   15750
УпдСчфДоп                 9226
ЭДОНакл                   5148
ЭДОСч                      372
АктВР                       55
ДетализацияАкта             54
ЭДОСчет                     45
Name: count, dtype: int64

In [102]:
dfs = []
for file in glob.glob('C:/Work/Analytics/Тест/Выгрузка данных в один источник/Входящие/*.csv'):
    df = pd.read_csv(file, sep=';', encoding='windows-1251')
    dfs.append(df)

sbis_df = pd.concat(dfs, ignore_index=True)
sbis_df.columns = [i.replace(' ', '_') for i in sbis_df.columns]
sbis_df.head(5)

,Дата,Номер,Сумма,Статус,Примечание,Комментарий,Контрагент,ИНН/КПП,Организация,ИНН/КПП.1,...,Время,Тип_пакета,Идентификатор_пакета,Запущено_в_обработку,Получено_контрагентом,Завершено,Увеличение_суммы,НДС,Уменьшение_суммы,НДС.1
0,30.09.21,БРН00709545,"4 056,46",Выполнение завершено успешно,NaN,"БРН00709545 на сумму 4 056,46р.","Пульс Брянск, ООО",3255510243 / 325701001,ООО Рога и Копыта,4025419873 / 402501001,...,23:40:56,ДокОтгрВх,76210df0-7c48-4e6d-ad2b-6cbe2f3b7722,30.09.21 23:40,01.10.21 08:39,01.10.21 14:59,"0,00","0,00","0,00","0,00"
1,30.09.21,БРН00709545,"4 056,46",Выполнение завершено успешно,NaN,"БРН00709545 на сумму 4 056,46р.","Пульс Брянск, ООО",3255510243 / 325701001,ООО Рога и Копыта,4025419873 / 402501001,...,23:40:56,ДокОтгрВх,76210df0-7c48-4e6d-ad2b-6cbe2f3b7722,NaN,NaN,NaN,"0,00","0,00","0,00","0,00"
2,30.09.21,БРН00709545,"0,00",Выполнение завершено успешно,NaN,"БРН00709545 на сумму 4 056,46р.","Пульс Брянск, ООО",3255510243 / 325701001,ООО Рога и Копыта,4025419873 / 402501001,...,23:40:56,ДокОтгрВх,76210df0-7c48-4e6d-ad2b-6cbe2f3b7722,NaN,NaN,NaN,"0,00","0,00","0,00","0,00"
3,30.09.21,БРН00709520,"54 705,65",Выполнение завершено успешно,NaN,"БРН00709520 на сумму 54 705,65р.","Пульс Брянск, ООО",3255510243 / 325701001,ООО Рога и Копыта,4025419873 / 402501001,...,23:34:51,ДокОтгрВх,1cfdcbd4-a43b-4543-90b6-59d0e6095b94,30.09.21 23:34,01.10.21 08:39,01.10.21 14:59,"0,00","0,00","0,00","0,00"
4,30.09.21,БРН00709520,"54 705,65",Выполнение завершено успешно,NaN,"БРН00709520 на сумму 54 705,65р.","Пульс Брянск, ООО",3255510243 / 325701001,ООО Рога и Копыта,4025419873 / 402501001,...,23:34:51,ДокОтгрВх,1cfdcbd4-a43b-4543-90b6-59d0e6095b94,NaN,NaN,NaN,"0,00","0,00","0,00","0,00"


аптеки

In [103]:
dfs_apteka = []
for file in glob.glob('C:/Work/Analytics/Примеры Тестовых/Выгрузка данных в один источник/Аптеки_csv_correct/*.csv'):
    dfs_apt = pd.read_csv(file, sep=';', encoding='windows-1251')
    dfs_apteka.append(dfs_apt)

In [104]:
dfs_apteka[0].columns

Index(['№ п/п', 'Штрих-код партии', 'Наименование товара', 'Поставщик',
       'Дата приходного документа', 'Номер приходного документа',
       'Дата накладной', 'Номер накладной', 'Кол-во',
       'Сумма в закупочных ценах без НДС', 'Ставка НДС поставщика',
       'Сумма НДС', 'Сумма в закупочных ценах с НДС'],
      dtype='str')

In [105]:
dfs_apteka[1].columns

Index(['№ п/п', 'Штрих-код партии', 'Наименование товара', 'Поставщик',
       'Дата приходного документа', 'Номер приходного документа',
       'Дата накладной', 'Номер накладной', 'Кол-во',
       'Сумма в закупочных ценах без НДС', 'Ставка НДС поставщика',
       'Сумма НДС', 'Сумма в закупочных ценах с НДС'],
      dtype='str')

In [106]:
dfs_apteka[0]['Номер накладной'].head(10)

0           4665120-30
1    E7914339/27806739
2        164621235-001
3        132215190-001
4        120939343-001
5           7596372-30
6             183390Р4
7        124924964-003
8           6289110-30
9        122696347-001
Name: Номер накладной, dtype: str

In [107]:
sbis_df[['Номер', 'Номер.1', 'Тип_документа']].head(10)

,Номер,Номер.1,Тип_документа
0,БРН00709545,БРН00709545,ЭДОНакл
1,БРН00709545,БРН00709545,СчФктр
2,БРН00709545,NaN,ЭДОРеестрСертификатов
3,БРН00709520,БРН00709520,ЭДОНакл
4,БРН00709520,БРН00709520,СчФктр
5,БРН00709520,NaN,ЭДОРеестрСертификатов
6,БРН00709520,NaN,ПротЦен
7,БРН00709519,БРН00709519,ЭДОНакл
8,БРН00709519,БРН00709519,СчФктр
9,БРН00709519,NaN,ЭДОРеестрСертификатов


In [108]:
numbers = dfs_apteka[0]['Номер накладной'].head(10)

sbis_df[
    sbis_df['Номер'].isin(numbers) |
    sbis_df['Номер.1'].isin(numbers)
][['Номер', 'Номер.1', 'Тип_документа', 'Дата', 'Сумма']].head(10)

,Номер,Номер.1,Тип_документа,Дата,Сумма
9662,7596372-30,NaN,ПротЦен,03.09.21,"0,00"
9663,7596372-30,7596372-30,СчФктр,03.09.21,"16 590,24"
9664,7596372-30,7596372-30,УпдДоп,03.09.21,"16 590,24"
9665,7596372-30,NaN,ЭДОРеестрСертификатов,03.09.21,"0,00"
10451,164621235-001,164621235-001,СчФктр,01.09.21,"13 129,99"
10452,164621235-001,164621235-001,УпдДоп,01.09.21,"13 129,99"
10453,164621235-001,NaN,ЭДОРеестрСертификатов,01.09.21,"0,00"
10454,164621235-001,NaN,ПротЦен,01.09.21,"0,00"
24860,6289110-30,6289110-30,СчФктр,20.07.21,"2 664,10"
24861,6289110-30,6289110-30,УпдДоп,20.07.21,"2 664,10"


Из огромного СБИС нам теперь нужны только:

Номер - по нему будем искать накладную

Сумма - ее перенесем в аптеку

Дата - ее тоже перенесем и потом сравним

In [109]:
doc_types = ['СчФктр', 'УпдДоп', 'УпдСчфДоп', 'ЭДОНакл']

sbis_filtered = sbis_df[
    sbis_df['Тип_документа'].isin(doc_types)
]

sbis_filtered[['Номер', 'Тип_документа', 'Дата', 'Сумма']].head(10)

,Номер,Тип_документа,Дата,Сумма
0,БРН00709545,ЭДОНакл,30.09.21,"4 056,46"
1,БРН00709545,СчФктр,30.09.21,"4 056,46"
3,БРН00709520,ЭДОНакл,30.09.21,"54 705,65"
4,БРН00709520,СчФктр,30.09.21,"54 705,65"
7,БРН00709519,ЭДОНакл,30.09.21,"5 890,95"
8,БРН00709519,СчФктр,30.09.21,"5 890,95"
11,БРН00709497,ЭДОНакл,30.09.21,"1 141,24"
12,БРН00709497,СчФктр,30.09.21,"1 141,24"
15,БРН00709476,ЭДОНакл,30.09.21,"5 822,85"
16,БРН00709476,СчФктр,30.09.21,"5 822,85"


In [110]:
sbis_filtered[
    sbis_filtered.duplicated(subset=['Номер'], keep=False)
].sort_values('Номер')[['Номер', 'Тип_документа', 'Дата', 'Сумма']]

,Номер,Тип_документа,Дата,Сумма
69566,00000010,УпдДоп,09.03.21,"648,00"
69565,00000010,СчФктр,09.03.21,"648,00"
69563,00000011,УпдДоп,09.03.21,"4 450,00"
69562,00000011,СчФктр,09.03.21,"4 450,00"
276754,00000849,ЭДОНакл,03.01.19,"20 993,53"
...,...,...,...,...
7,БРН00709519,ЭДОНакл,30.09.21,"5 890,95"
3,БРН00709520,ЭДОНакл,30.09.21,"54 705,65"
4,БРН00709520,СчФктр,30.09.21,"54 705,65"
1,БРН00709545,СчФктр,30.09.21,"4 056,46"


In [111]:
sbis_filtered = sbis_filtered.drop_duplicates(subset=['Номер', 'Дата', 'Сумма'])

In [112]:
sbis_filtered.shape

(88407, 30)

In [113]:
sbis_filtered[sbis_filtered.duplicated(subset=['Номер'], keep=False)]

,Дата,Номер,Сумма,Статус,Примечание,Комментарий,Контрагент,ИНН/КПП,Организация,ИНН/КПП.1,...,Время,Тип_пакета,Идентификатор_пакета,Запущено_в_обработку,Получено_контрагентом,Завершено,Увеличение_суммы,НДС,Уменьшение_суммы,НДС.1
38,30.09.21,8561768-30,"18 073,24",Выполнение завершено успешно,NaN,NaN,"Филиал АО НПК ""Катрен"" в г. Химки",5408130693 / 504743001,ООО Рога и Копыта,4025419873 / 402501001,...,23:03:54,ДокОтгрВх,9d07ae2d-b294-497b-8877-ba827ab2d42d,NaN,NaN,NaN,"0,00","0,00","0,00","0,00"
39,30.09.21,8561768-30,"18 108,44",Выполнение завершено успешно,NaN,NaN,"Филиал АО НПК ""Катрен"" в г. Химки",5408130693 / 504743001,ООО Рога и Копыта,4025419873 / 402501001,...,23:03:54,ДокОтгрВх,9d07ae2d-b294-497b-8877-ba827ab2d42d,NaN,NaN,NaN,"0,00","0,00","0,00","0,00"
550,29.09.21,8510179-30,"1 046,98",Выполнение завершено успешно,NaN,Интернет заказ: AM-49567708; Интернет заказ: A...,"Филиал АО НПК ""Катрен"" в г. Химки",5408130693 / 504743001,ООО Рога и Копыта,4025419873 / 402501001,...,14:07:33,ДокОтгрВх,2136e203-2bca-4d48-ac48-d1dc738e0cf9,29.09.21 14:07,29.09.21 17:18,30.09.21 13:12,"0,00","0,00","0,00","0,00"
930,28.09.21,8475851-30,"41,86",Выполнение завершено успешно,NaN,Интернет заказ: AM-49483950; Интернет заказ: A...,"Филиал АО НПК ""Катрен"" в г. Химки",5408130693 / 504743001,ООО Рога и Копыта,4025419873 / 402501001,...,18:19:36,ДокОтгрВх,f8cf7d01-b044-42ea-ae55-d519bb49124f,28.09.21 18:19,29.09.21 10:46,29.09.21 10:58,"0,00","0,00","0,00","0,00"
931,28.09.21,8475851-30,"318,69",Выполнение завершено успешно,NaN,Интернет заказ: AM-49483950; Интернет заказ: A...,"Филиал АО НПК ""Катрен"" в г. Химки",5408130693 / 504743001,ООО Рога и Копыта,4025419873 / 402501001,...,18:19:36,ДокОтгрВх,f8cf7d01-b044-42ea-ae55-d519bb49124f,NaN,NaN,NaN,"0,00","0,00","0,00","0,00"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
295357,02.10.18,2833508-30,"8 833,89",Выполнение завершено успешно,NaN,NaN,"Филиал АО НПК ""Катрен"" в г. Химки",5408130693 / 504743001,ООО Рога и Копыта,4025419873 / 402501001,...,23:01:40,ДокОтгрВх,4e072fb7-903e-4f49-aed7-bd4600444c12,NaN,NaN,NaN,"0,00","0,00","0,00","0,00"
295366,02.10.18,00397773,"3 178,18",Удален контрагентом,NaN,NaN,"Норман, ООО",3625010992 / 362501001,ООО Рога и Копыта,4025419873 / 402501001,...,21:59:00,ДокОтгрВх,0f633403-2105-4ad8-a0b0-08c3d70a5e52,03.10.18 13:24,03.10.18 15:21,03.10.18 15:25,"0,00","0,00","0,00","0,00"
295587,01.10.18,БРН00367741,"6 745,65",Выполнение завершено успешно,NaN,NaN,"Пульс Брянск, ООО",3255510243 / 325701001,ООО Рога и Копыта,4025419873 / 402501001,...,21:52:00,ДокОтгрВх,950aff3f-3a87-4ac1-aa70-3658b50ca1ca,01.10.18 21:51,02.10.18 10:16,02.10.18 11:14,"0,00","0,00","0,00","0,00"
295619,01.10.18,2815323-30,"6 701,81",Выполнение завершено успешно,NaN,NaN,"Филиал АО НПК ""Катрен"" в г. Химки",5408130693 / 504743001,ООО Рога и Копыта,4025419873 / 402501001,...,19:45:00,ДокОтгрВх,6e6b135a-0f44-4c12-bedc-79b053f0477f,01.10.18 19:44,02.10.18 10:16,02.10.18 12:21,"0,00","0,00","0,00","0,00"


После удаления одинаковых записей остались случаи, когда одному номеру соответствуют несколько записей с разными суммами. В условии не указано, какую запись выбирать, поэтому оставляем первую найденную

In [114]:
sbis_unique = sbis_filtered.drop_duplicates(
    subset=['Номер'],
    keep='first'
)

#без sbis_unique merge размножит аптечные строки. правило выбора между разными суммами автор задания не дал

In [115]:
df_merged = dfs_apteka[0].merge(
    sbis_unique[['Номер', 'Дата', 'Сумма']],
    left_on='Номер накладной',
    right_on='Номер',
    how='left'
)

In [116]:
dfs_apteka[0].shape

(5039, 13)

In [117]:
df_merged.shape

(5039, 16)

In [118]:
df_merged[df_merged.duplicated(subset=['№ п/п'], keep=False)].head(5)

,№ п/п,Штрих-код партии,Наименование товара,Поставщик,Дата приходного документа,Номер приходного документа,Дата накладной,Номер накладной,Кол-во,Сумма в закупочных ценах без НДС,Ставка НДС поставщика,Сумма НДС,Сумма в закупочных ценах с НДС,Номер,Дата,Сумма


In [119]:
mask = dfs_apteka[1]['Поставщик'].str.contains('ЕАПТЕКА', na = False)

dfs_apteka[1].loc[mask, 'Номер накладной'] = (dfs_apteka[1].loc[mask, 'Номер накладной'].astype(str) + '/15')

dfs_apteka[1].loc[mask, ['Поставщик', 'Номер накладной']]

,Поставщик,Номер накладной
250,ЕАПТЕКА ООО,11724317 Апт.1/15
1061,ЕАПТЕКА ООО,ЦЕ000193605/15
1208,ЕАПТЕКА ООО,14901893/15
1772,ЕАПТЕКА ООО,15686437/15
1773,ЕАПТЕКА ООО,15686437/15
1774,ЕАПТЕКА ООО,15688830/15
1775,ЕАПТЕКА ООО,15688830/15
1776,ЕАПТЕКА ООО,15688830/15
1846,ЕАПТЕКА ООО,15733509/15
1847,ЕАПТЕКА ООО,15733509/15


In [120]:
df_merged = df_merged.rename(columns={'Номер': 'Номер счет-фактуры',
    'Сумма': 'Сумма счет-фактуры',
    'Дата': 'Дата счет-фактуры'})


In [121]:
df_merged.head()

,№ п/п,Штрих-код партии,Наименование товара,Поставщик,Дата приходного документа,Номер приходного документа,Дата накладной,Номер накладной,Кол-во,Сумма в закупочных ценах без НДС,Ставка НДС поставщика,Сумма НДС,Сумма в закупочных ценах с НДС,Номер счет-фактуры,Дата счет-фактуры,Сумма счет-фактуры
0,1,200200060311,СФМ ПЕРЧАТКИ СМОТР. Н/СТЕР. ЛАТ. Р.S №100 (50П...,Катрен г.Химки,26.05.2021,2002088.0,25.05.2021,4665120-30,0.040000,33.65,10%,3.37,37.02,4665120-30,25.05.21,"16 728,87"
1,2,200801594420,КОСМОПОР ПОВЯЗКА 10СМХ6СМ №25 П/ОПЕР. САМОКЛ. ...,Здравсервис,25.03.2020,9000479.8,25.03.2020,E7914339/27806739,0.040000,27.58,10%,2.76,30.34,E7914339/27806739,25.03.20,"3 395,94"
2,3,200200075091,ХАРТМАНН БРАНОЛИНД H ПОВЯЗКА СТЕР. 10Х20СМ. №3...,Протек,01.09.2021,2003451.0,01.09.2021,164621235-001,0.066667,145.28,10%,14.53,159.81,164621235-001,01.09.21,"13 129,99"
3,4,200801670938,КОСМОПОР Е ПОВЯЗКА 20СМХ8СМ №25 П/ОПЕР. САМОКЛ...,Протек,17.06.2020,9000628.8,17.06.2020,132215190-001,0.080000,47.07,10%,4.71,51.78,132215190-001,17.06.20,"11 893,01"
4,5,200801489808,"ФОКУСИН 0,4МГ. №90 КАПС. МОДИФ.ВЫСВ. /ЗЕНТИВА/...",Протек,12.12.2019,9001590.8,12.12.2019,120939343-001,0.111111,96.37,10%,9.64,106.00,120939343-001,12.12.19,"3 219,81"


In [122]:
df_merged['Дата накладной'] = pd.to_datetime(df_merged['Дата накладной'], format = 'mixed',
    dayfirst=True, errors="coerce")

df_merged['Дата счет-фактуры'] = pd.to_datetime(df_merged['Дата счет-фактуры'], format = 'mixed',
    dayfirst=True, errors="coerce")

In [123]:
df_merged.dtypes

№ п/п                                        int64
Штрих-код партии                             int64
Наименование товара                            str
Поставщик                                      str
Дата приходного документа                      str
Номер приходного документа                 float64
Дата накладной                      datetime64[us]
Номер накладной                                str
Кол-во                                     float64
Сумма в закупочных ценах без НДС           float64
Ставка НДС поставщика                          str
Сумма НДС                                  float64
Сумма в закупочных ценах с НДС             float64
Номер счет-фактуры                             str
Дата счет-фактуры                   datetime64[us]
Сумма счет-фактуры                             str
dtype: object

In [124]:
mask = (
    df_merged['Дата счет-фактуры'].notna()
    &
    (df_merged['Дата накладной'] != df_merged['Дата счет-фактуры'])
)

df_merged['Сравнение дат'] = ''
df_merged.loc[mask, 'Сравнение дат'] = 'Не совпадает!'

In [125]:
df_merged['Дата счет-фактуры'] = (df_merged['Дата счет-фактуры'].dt.strftime('%d.%m.%Y'))

In [126]:
df_merged['Сравнение дат'].value_counts(dropna=False)

Сравнение дат
                 4868
Не совпадает!     171
Name: count, dtype: int64

In [127]:
columns = ['№ п/п', 'Штрих-код партии', 'Наименование товара', 'Поставщик',
    'Дата приходного документа', 'Номер приходного документа',
    'Дата накладной', 'Номер накладной', 'Номер счет-фактуры',
    'Сумма счет-фактуры', 'Кол-во',
    'Сумма в закупочных ценах без НДС', 'Ставка НДС поставщика',
    'Сумма НДС', 'Сумма в закупочных ценах с НДС',
    'Дата счет-фактуры', 'Сравнение дат']

In [128]:
df_merged = df_merged[columns]

In [129]:
df_merged

,№ п/п,Штрих-код партии,Наименование товара,Поставщик,Дата приходного документа,Номер приходного документа,Дата накладной,Номер накладной,Номер счет-фактуры,Сумма счет-фактуры,Кол-во,Сумма в закупочных ценах без НДС,Ставка НДС поставщика,Сумма НДС,Сумма в закупочных ценах с НДС,Дата счет-фактуры,Сравнение дат
0,1,200200060311,СФМ ПЕРЧАТКИ СМОТР. Н/СТЕР. ЛАТ. Р.S №100 (50П...,Катрен г.Химки,26.05.2021,2002088.0,2021-05-25,4665120-30,4665120-30,"16 728,87",0.040000,33.65,10%,3.37,37.02,25.05.2021,
1,2,200801594420,КОСМОПОР ПОВЯЗКА 10СМХ6СМ №25 П/ОПЕР. САМОКЛ. ...,Здравсервис,25.03.2020,9000479.8,2020-03-25,E7914339/27806739,E7914339/27806739,"3 395,94",0.040000,27.58,10%,2.76,30.34,25.03.2020,
2,3,200200075091,ХАРТМАНН БРАНОЛИНД H ПОВЯЗКА СТЕР. 10Х20СМ. №3...,Протек,01.09.2021,2003451.0,2021-09-01,164621235-001,164621235-001,"13 129,99",0.066667,145.28,10%,14.53,159.81,01.09.2021,
3,4,200801670938,КОСМОПОР Е ПОВЯЗКА 20СМХ8СМ №25 П/ОПЕР. САМОКЛ...,Протек,17.06.2020,9000628.8,2020-06-17,132215190-001,132215190-001,"11 893,01",0.080000,47.07,10%,4.71,51.78,17.06.2020,
4,5,200801489808,"ФОКУСИН 0,4МГ. №90 КАПС. МОДИФ.ВЫСВ. /ЗЕНТИВА/...",Протек,12.12.2019,9001590.8,2019-12-12,120939343-001,120939343-001,"3 219,81",0.111111,96.37,10%,9.64,106.00,12.12.2019,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5034,5035,200801289501,Карта LOYALITY 25Р,"ООО ""Континент""",06.05.2019,8000087.0,2019-05-06,06.05.2019,NaN,NaN,63.000000,1575.00,NaN,0.00,1575.00,NaN,
5035,5036,200801680549,ПАКЕТ МАЙКА №1,Подотчетное лицо,27.01.2021,8000018.0,2021-01-27,NaN,NaN,NaN,67.000000,113.90,NaN,0.00,113.90,NaN,
5036,5037,200200000519,БУТЫЛОЧКА МОЛОЧНАЯ 200МЛ.,АйТи-Аптека Внедрение,15.03.2011,2000029.0,2011-03-15,NaN,NaN,NaN,80.000000,369.60,18%,66.40,436.00,NaN,
5037,5038,200200079475,КОРВАЛОЛ 25МЛ. КАПЛИ И/У /ФАРМСТАНДАРТ ЛЕКСРЕД...,Пульс,24.09.2021,2003811.0,2021-09-23,БРН00689248,БРН00689248,"40 260,90",113.000000,1880.32,10%,187.58,2067.90,23.09.2021,


In [130]:
df_merged.shape

(5039, 17)

Для второй аптеки делаем то же самое, только у неё уже исправлены номера ЕАПТЕКИ

In [131]:
df_merged_2 = dfs_apteka[1].merge(sbis_unique[['Номер', 'Дата', 'Сумма']], left_on = 'Номер накладной', right_on = 'Номер', how = 'left')

df_merged_2 = df_merged_2.rename(columns={'Номер': 'Номер счет-фактуры',
    'Сумма': 'Сумма счет-фактуры',
    'Дата': 'Дата счет-фактуры'})

df_merged_2['Дата накладной'] = pd.to_datetime(df_merged_2['Дата накладной'], format = 'mixed', dayfirst=True, errors="coerce")

df_merged_2['Дата счет-фактуры'] = pd.to_datetime(df_merged_2['Дата счет-фактуры'], format = 'mixed', dayfirst=True, errors="coerce")

mask = (df_merged_2['Дата счет-фактуры'].notna() &
        (df_merged_2['Дата накладной'] != df_merged_2['Дата счет-фактуры']))

df_merged_2['Сравнение дат'] = ''
df_merged_2.loc[mask, 'Сравнение дат'] = 'Не совпадает!'

In [132]:
df_merged_2 = df_merged_2[columns]

In [133]:
df_merged_2.shape

(4819, 17)

In [140]:
df_merged.to_excel('apteka_1.xlsx', index=False)

In [141]:

df_merged_2.to_excel('apteka_2.xlsx', index=False)